In [30]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# load the dataset from the raw_data folder
df = pd.read_csv('raw_data/sales_data.csv')
print(df)

,date,sku_id,product_name,category,unit_price,inventory_level,promotion,is_weekend,units_sold
0,2026-06-01,SKU_001,Organic Bananas,Fruit,1.95,107.0,No,0,91
1,2026-06-01,SKU_002,Baby Spinach,Leafy Greens,3.47,86.0,Yes,0,66
2,2026-06-01,SKU_003,Roma Tomatoes,Vegetables,2.40,85.0,No,0,55
3,2026-06-01,SKU_004,Hass Avocados,Fruit,5.03,84.0,No,0,46
4,2026-06-01,SKU_005,Organic Gala Apples,Fruit,NaN,85.0,NaN,0,48
...,...,...,...,...,...,...,...,...,...
1099,2026-08-31,SKU_008,Red Bell Peppers,Vegetables,3.05,61.0,No,0,41
1100,2026-08-31,SKU_009,English Cucumbers,Vegetables,1.65,49.0,No,0,34
1101,2026-08-31,SKU_010,Organic Carrots,Vegetables,2.24,67.0,No,0,55
1102,2026-08-31,SKU_011,Fresh Blueberries,Berries,5.08,45.0,No,0,25


In [ ]:
print(df.head())
print("================================")
print(df.info())
print("================================")
print(df.describe())
print("================================")
print(df.isnull().sum())

         date   sku_id         product_name      category  unit_price  \
0  2026-06-01  SKU_001      Organic Bananas         Fruit        1.95   
1  2026-06-01  SKU_002         Baby Spinach  Leafy Greens        3.47   
2  2026-06-01  SKU_003        Roma Tomatoes    Vegetables        2.40   
3  2026-06-01  SKU_004        Hass Avocados         Fruit        5.03   
4  2026-06-01  SKU_005  Organic Gala Apples         Fruit         NaN   

   inventory_level promotion  is_weekend  units_sold  
0            107.0        No           0          91  
1             86.0       Yes           0          66  
2             85.0        No           0          55  
3             84.0        No           0          46  
4             85.0       NaN           0          48  
<class 'pandas.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             1104 non-null   str   

In [22]:
# find categorical column olny
categorial_columns = df.describe(include=['object']).columns
print("Categorical Columns:", categorial_columns)

Categorical Columns: Index(['date', 'sku_id', 'product_name', 'category', 'promotion'], dtype='str')


C:\Users\Mushtaq Ahmad Madni\AppData\Local\Temp\ipykernel_10244\3900425169.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorial_columns = df.describe(include=['object']).columns


In [23]:
# find numerical column only
numerical_columns = df.describe(include=['number']).columns
print("Numerical Columns:", numerical_columns)

Numerical Columns: Index(['unit_price', 'inventory_level', 'is_weekend', 'units_sold'], dtype='str')


In [26]:
# find only null values in the dataset
null_values = df.isnull().sum()
print("Null Values in the Dataset:\n", null_values[null_values > 0])
print("================================")
# show only datatye of null_values
print("Data Types of Null Values:\n", df.dtypes[null_values > 0])

Null Values in the Dataset:
 unit_price         44
inventory_level    33
promotion          33
dtype: int64
Data Types of Null Values:
 unit_price         float64
inventory_level    float64
promotion              str
dtype: object


In [ ]:
# Sort according to date
df = df.sort_values("date").reset_index(drop=True)

# Bina shuffle kiye time-based split karein
n = len(df) # Yeh dataset ki total rows ginta hai.
train_set = df.iloc[: int(0.7 * n)].copy()
val_set = df.iloc[int(0.7 * n) : int(0.9 * n)].copy()
test_set = df.iloc[int(0.9 * n) :].copy()

In [ ]:
# Save The data in split data folder

# 1. Folder ka naam define karein aur agar mojood nahi toh create karein
output_dir = "split_data"
os.makedirs(output_dir, exist_ok=True)

# 2. Files ko 'split_data' folder ke andar save karein
train_path = os.path.join(output_dir, "train.csv")
val_path = os.path.join(output_dir, "val.csv")
test_path = os.path.join(output_dir, "test.csv")

train_set.to_csv(train_path, index=False)
val_set.to_csv(val_path, index=False)
test_set.to_csv(test_path, index=False)

print(f"Files kamyabi se '{output_dir}' folder mein save ho chuki hain:")
print(f" - {train_path}")
print(f" - {val_path}")
print(f" - {test_path}")

Files kamyabi se 'split_data' folder mein save ho chuki hain:
 - split_data\train.csv
 - split_data\val.csv
 - split_data\test.csv


In [32]:
# 1. Split kiye huye data ko load karein
train_df = pd.read_csv("split_data/train.csv")
val_df = pd.read_csv("split_data/val.csv")
test_df = pd.read_csv("split_data/test.csv")

# 2. Features alag karein jinko impute karna hai
num_cols = ["unit_price", "inventory_level"]
cat_cols = ["promotion"]

# 3. Pipelines define karein (Numeric ke liye median, Categorical ke liye most_frequent)
num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

cat_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))])

# 4. Helper function: Statistics sirf TRAIN se fit karega aur teeno par transform karega
def impute_and_update(train, val, test):
  # Fit sirf train par
  num_pipeline.fit(train[num_cols])
  cat_pipeline.fit(train[cat_cols])

  # Teeno splits par values apply karein
  for df_split in [train, val, test]:
    df_split[num_cols] = num_pipeline.transform(df_split[num_cols])
    df_split[cat_cols] = cat_pipeline.transform(df_split[cat_cols])

  return train, val, test


# 5. Imputation run karein
train_df, val_df, test_df = impute_and_update(train_df, val_df, test_df)

# 7. Saaf (cleaned) data ko wapis files mein overwrite kar dein
train_df.to_csv("split_data/train.csv", index=False)
val_df.to_csv("split_data/val.csv", index=False)
test_df.to_csv("split_data/test.csv", index=False)

In [33]:
# 6. Check karein ke koi missing value toh nahi bachi
print(
    "Train missing values bachi hain:", train_df[num_cols + cat_cols].isnull().sum().to_dict()
)
print(
    "Val missing values bachi hain:  ", val_df[num_cols + cat_cols].isnull().sum().to_dict()
)
print(
    "Test missing values bachi hain: ", test_df[num_cols + cat_cols].isnull().sum().to_dict()
)

print("\nKamyabi se teeno files clean ho kar save ho chuki hain!")

Train missing values bachi hain: {'unit_price': 0, 'inventory_level': 0, 'promotion': 0}
Val missing values bachi hain:   {'unit_price': 0, 'inventory_level': 0, 'promotion': 0}
Test missing values bachi hain:  {'unit_price': 0, 'inventory_level': 0, 'promotion': 0}

Kamyabi se teeno files clean ho kar save ho chuki hain!


In [1]:
import pandas as pd

train_df = pd.read_csv("split_data/train.csv")
val_df = pd.read_csv("split_data/val.csv")
test_df = pd.read_csv("split_data/test.csv")

print("Train Missing Values:\n", train_df.isnull().sum())
print("-" * 35)
print("Val Missing Values:\n", val_df.isnull().sum())
print("-" * 35)
print("Test Missing Values:\n", test_df.isnull().sum())

Train Missing Values:
 date               0
sku_id             0
product_name       0
category           0
unit_price         0
inventory_level    0
promotion          0
is_weekend         0
units_sold         0
dtype: int64
-----------------------------------
Val Missing Values:
 date               0
sku_id             0
product_name       0
category           0
unit_price         0
inventory_level    0
promotion          0
is_weekend         0
units_sold         0
dtype: int64
-----------------------------------
Test Missing Values:
 date               0
sku_id             0
product_name       0
category           0
unit_price         0
inventory_level    0
promotion          0
is_weekend         0
units_sold         0
dtype: int64
